# 10. AI Evaluation Science: Bradley-Terry, Item Response Theory, Dawid-Skene, and Min-K%

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Bradley & Terry (1952 *Rank analysis of incomplete block designs*); Lord (1980 *Applications of item response theory*); Dawid & Skene (1979 *Maximum likelihood estimation of observer error-rates*); Shi et al. (2023 *Detecting Pretraining Data from Large Language Models by Min-K% Prob*).
- **Purpose:** Implement rigorous AI psychometrics and evaluation pipelines: Bradley-Terry-Luce preference modeling, 2PL Item Response Theory for question difficulty calibration, Dawid-Skene annotator consensus EM, and Min-K% Prob pretraining data extraction detection.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [09. AI Analysis Loss Landscape Hessian Lanczos and Empirical NTK](09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.ipynb) $\to$ `🔗 Downstream Syntheses:` [11. Statistical Model Comparison Bayesian RoPE Friedman and Bootstrap](11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not fit a unidimensional BTL model across conflicting multidimensional domains (intransitive rock-paper-scissors loops distort skill estimates); avoid raw Min-K% on highly repetitive prompts without length and entropy normalization.

## Mathematical Foundations & Governing Equations

1. **Bradley-Terry-Luce (BTL)**: $P(i \succ j) = \sigma(s_i - s_j) = \frac{1}{1 + e^{-(s_i - s_j)}}$
2. **2PL Item Response Theory**: $P(Y_{mi}=1) = \sigma(a_i(\theta_m - b_i)) = \frac{1}{1 + e^{-a_i(\theta_m - b_i)}}$
3. **Dawid-Skene E-Step**: $T_{ik} \propto p_k \prod_{j=1}^J \Pi_{k, Y_{ij}}^{(j)}$
4. **Min-K% Prob**: $\text{Min-K\% Prob} = \frac{1}{|K_{\min}|} \sum_{t \in K_{\min}} \ln P(x_t)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_btl_win_prob(skills):
    diff = skills.unsqueeze(1) - skills.unsqueeze(0)
    return torch.sigmoid(diff)

def calc_irt_2pl(abilities, difficulties, discriminations):
    diff = abilities.unsqueeze(1) - difficulties.unsqueeze(0)
    return torch.sigmoid(discriminations.unsqueeze(0) * diff)

def calc_dawid_skene(annotations, priors, confusion):
    n, j_cnt = annotations.shape
    k_cnt = priors.shape[0]
    log_p = torch.zeros(n, k_cnt)
    for i in range(n):
        for k in range(k_cnt):
            lp = torch.log(priors[k] + 1e-8)
            for j in range(j_cnt):
                obs = int(annotations[i, j].item())
                lp = lp + torch.log(confusion[j, k, obs] + 1e-8)
            log_p[i, k] = lp
    return torch.softmax(log_p, dim=-1)

def calc_min_k(token_log_probs, k_fraction=0.25):
    sorted_lp, _ = torch.sort(token_log_probs)
    k = max(1, int(round(k_fraction * token_log_probs.shape[0])))
    return sorted_lp[:k].mean()

In [ ]:
# Parity verification against golden reference tensors
btl_p = calc_btl_win_prob(weights["btl_skills"])
assert torch.max(torch.abs(btl_p - weights["btl_win_prob_ref"])) < 1e-5, "BTL mismatch"

irt_p = calc_irt_2pl(weights["irt_abilities"], weights["irt_difficulties"], weights["irt_discriminations"])
assert torch.max(torch.abs(irt_p - weights["irt_prob_ref"])) < 1e-5, "IRT mismatch"

ds_post = calc_dawid_skene(weights["ds_annotations"], weights["ds_priors"], weights["ds_confusion"])
assert torch.max(torch.abs(ds_post - weights["ds_posterior_ref"])) < 1e-5, "Dawid-Skene mismatch"

mink = calc_min_k(weights["mink_log_probs"], 0.25)
assert torch.max(torch.abs(mink - weights["mink_score_ref"])) < 1e-5, "Min-K% mismatch"
print("All BTL, IRT, Dawid-Skene, and Min-K% assertions passed!")

In [ ]:
# Benchmarking evaluation science operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_btl_win_prob(weights["btl_skills"])
    _ = calc_irt_2pl(weights["irt_abilities"], weights["irt_difficulties"], weights["irt_discriminations"])
    _ = calc_dawid_skene(weights["ds_annotations"], weights["ds_priors"], weights["ds_confusion"])
    _ = calc_min_k(weights["mink_log_probs"], 0.25)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused BTL + IRT + DawidSkene + Min-K% latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Detecting Pretraining Memorization vs Unseen Text via Min-K% Prob
# Memorized sequence: even the most surprising tokens have high log-probability
memorized_lp = torch.tensor([-0.05, -0.10, -0.25, -0.40, -0.15, -0.30, -0.35, -0.20])
# Unseen non-member text: contains outlier low-probability surprise tokens
unseen_lp = torch.tensor([-0.05, -0.10, -2.85, -0.40, -0.15, -3.40, -0.35, -0.20])

score_mem = calc_min_k(memorized_lp, k_fraction=0.25).item()
score_unseen = calc_min_k(unseen_lp, k_fraction=0.25).item()
print(f"Memorized Min-K% score: {score_mem:.3f} vs Unseen Min-K% score: {score_unseen:.3f}")
assert score_mem > -0.5, "Memorized text retains high log-probability across all tokens!"
assert score_unseen < -2.5, "Unseen text exhibits deep negative outlier log-probabilities!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Bradley-Terry-Luce** transforms pairwise comparative feedback into calibrated scalar abilities and Elo ratings.
2. **Item Response Theory** disentangles model capability from question difficulty and discrimination.
3. **Dawid-Skene Consensus EM** aggregates multi-judge annotations while estimating individual rater confusion matrices.
4. **Min-K% Prob** effectively flags benchmark contamination and pretraining memorization without training external classifiers.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.md](../../markdown/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.md)
- **PyTorch 2.x Module:** [10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.py](../../pytorch2/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.py)
- **Functional JAX Module:** [10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.py](../../jax/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.py)
- **Native Rust Candle Track:** [10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.rs](../../candle/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.rs)
- **Hardware Kernel (CPU / WGSL):** [10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.rs](../../kernels/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.rs)
- **Golden SafeTensors:** [10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.safetensors](../../weights/10_xai_and_ai_analysis_methods/10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 09. AI Analysis Loss Landscape Hessian Lanczos and Empirical NTK](09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 11. Statistical Model Comparison Bayesian RoPE Friedman and Bootstrap](11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.ipynb) &rarr;